# 📊 UnthAI Annotation Dashboard
This notebook provides a real-time snapshot of the annotation progress and label distributions in `annotation_part_1.csv`.

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import os

# Settings
FILE_PATH = "annotation_part_1.csv"
CATEGORIES = ['food', 'service', 'place', 'delivery', 'price', 'treatment']

if not os.path.exists(FILE_PATH):
    print(f"❌ File {FILE_PATH} not found.")
else:
    df = pd.read_csv(FILE_PATH, keep_default_na=False)
    print(f"✅ Loaded {len(df)} rows from {FILE_PATH}")

✅ Loaded 6197 rows from annotation_part_1.csv


## 1. Overall Progress

In [22]:
# A row is considered labeled if 'out_of_scope' is True or False
labeled_mask = df['out_of_scope'].isin(['True', 'False', True, False])
total_labeled = labeled_mask.sum()
total_rows = len(df)
remaining = total_rows - total_labeled
progress = (total_labeled / total_rows) * 100

# Prettier HTML Progress Bar
html_str = f"""
<div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; padding: 3px; box-shadow: inset 0 1px 3px rgba(0,0,0,.2);">
  <div style="width: {progress}%; height: 30px; background-color: #2196f3; border-radius: 7px; transition: width 0.5s ease-in-out; display: flex; align-items: center; justify-content: center; color: white; font-family: sans-serif; font-weight: bold;">
    {progress:.1f}%
  </div>
</div>
<div style="display: flex; justify-content: space-between; margin-top: 10px; font-family: sans-serif; font-size: 14px;">
  <span>✅ <b>Labeled:</b> {total_labeled}</span>
  <span>⏳ <b>Remaining:</b> {remaining}</span>
  <span>📌 <b>Total:</b> {total_rows}</span>
</div>
"""

display(HTML(html_str))

## 2. Intent Distribution (Blue Styled Table)

In [23]:
# Pre-processing: handle empty strings and unify casing
for cat in CATEGORIES:
    df[cat] = df[cat].apply(lambda x: str(x).strip().capitalize())
    df[cat] = df[cat].replace({'': 'None', 'Nan': 'None', 'nan': 'None'})

# Aggregate counts for Table
table_data = []
for cat in CATEGORIES:
    counts = df[labeled_mask][cat].value_counts()
    counts.name = cat
    table_data.append(counts)

distribution_table = pd.concat(table_data, axis=1).fillna(0).astype(int).T

# Remove 'None' and add 'Total'
if 'None' in distribution_table.columns:
    distribution_table = distribution_table.drop(columns=['None'])

distribution_table['Total'] = distribution_table.sum(axis=1)

print("\n--- Intent Distribution (Total per Category) ---")
# Styled table: blue/white gradient
distribution_table.style.background_gradient(cmap='Blues', axis=None)


--- Intent Distribution (Total per Category) ---


,Appreciation,Complaint,Inquiry,Recommendation,"Appreciation,complaint","Complaint,recommendation","Complaint,inquiry","Appreciation,inquiry","Appreciation,recommendation","Inquiry,recommendation",Total
food,1201,691,84,60,17,2,2,1,1,0,2059
service,622,230,1,1,0,0,0,0,0,0,854
place,1372,431,549,32,1,1,6,2,0,1,2395
delivery,12,19,141,1,0,0,0,0,0,0,173
price,178,707,195,10,0,0,3,0,0,1,1094
treatment,752,248,5,7,0,0,0,1,0,0,1013


## 3. High-Level Insights

In [24]:
oos_counts = df[labeled_mask]['out_of_scope'].astype(str).value_counts()
print("--- Out of Scope Distribution ---")
print(oos_counts)

# Multi-label stats
def count_labels(row):
    return sum(1 for cat in CATEGORIES if row[cat] != 'None')

df['label_count'] = df[labeled_mask].apply(count_labels, axis=1)
multi_label_counts = df['label_count'].value_counts().sort_index()

print("\n--- Multi-Label Engagement ---")
for count, freq in multi_label_counts.items():
    print(f"{int(count)} categories tagged: {freq} comments")

--- Out of Scope Distribution ---
out_of_scope
False    4200
True     1997
Name: count, dtype: int64

--- Multi-Label Engagement ---
0 categories tagged: 1997 comments
1 categories tagged: 2630 comments
2 categories tagged: 594 comments
3 categories tagged: 249 comments
4 categories tagged: 613 comments
5 categories tagged: 113 comments
6 categories tagged: 1 comments
